# Ülgen — kendi sesimle Piper modeli eğitimi

Bu defter Google Colab'ın **ücretsiz GPU**'sunda çalışır. Ülgen'in *Kendi sesimi kaydet* sayfasından indirdiğiniz `ses_veri.zip` dosyasıyla, hazır Türkçe Piper sesinden (dfki) başlayarak sizin sesinize uyarlanmış bir model üretir.

**Başlamadan:** Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seçin. Sonra hücreleri sırayla ▶ ile çalıştırın.

Eğitim birkaç saat sürer. Ara verilirse kaldığı yerden sürsün diye her şey Google Drive'a kaydedilir.

In [ ]:
# 1) GPU var mı?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Google Drive'ı bağla (eğitim yarıda kesilirse buradan sürer)
from google.colab import drive
drive.mount('/content/drive')
KOK = '/content/drive/MyDrive/ulgen_ses'
import os; os.makedirs(KOK, exist_ok=True); print(KOK)

In [ ]:
# 3) Piper eğitim araçlarını kur (5-10 dakika)
!apt-get -qq install -y build-essential cmake ninja-build espeak-ng > /dev/null
!rm -rf /content/piper1-gpl && git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py -q build_ext --inplace
%cd /content

In [ ]:
# 4) ses_veri.zip dosyanızı yükleyin (Ülgen > Ayarlar > Ses > Kendi sesimi kaydet > Eğitim paketini indir)
import os, zipfile
if not os.path.exists(KOK + '/ses_veri.zip'):
    from google.colab import files
    yuklenen = files.upload()
    ad = next(iter(yuklenen))
    os.replace(ad, KOK + '/ses_veri.zip')
with zipfile.ZipFile(KOK + '/ses_veri.zip') as z:
    z.extractall('/content/veri')
satir = sum(1 for _ in open('/content/veri/metadata.csv', encoding='utf-8'))
print(satir, 'cümle yüklendi')

In [ ]:
# 5) Başlangıç noktası: hazır Türkçe ses (dfki, orta kalite) eğitim dosyası (~850 MB, bir kez)
CKPT = KOK + '/dfki.ckpt'
if not os.path.exists(CKPT):
    !wget -q --show-progress -O "$CKPT" "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/tr/tr_TR/dfki/medium/epoch%3D5679-step%3D1489110.ckpt"
print(os.path.getsize(CKPT)//1_000_000, 'MB')

## 6) Eğitim

`EK_TUR`: dfki'nin üstüne kaç tur daha eğitileceği. 500 cümle için 1000 tur ≈ 2–4 saat (T4). Colab sizi atarsa bu hücreyi yeniden çalıştırın: Drive'daki son kayıttan sürer.

Başlangıç dosyası yüklenemezse (sürüm uyuşmazlığı hatası), aşağıdaki `BASLAT` değerini `'vocoder'` yapın: o zaman yalnız ses üretici kısım alınır, eğitim biraz daha uzun sürer.

In [ ]:
EK_TUR = 1000
BASLAT = 'ckpt'   # 'ckpt' ya da 'vocoder'
import glob
SES_ADI = 'tr_TR-hakan-medium'
son = sorted(glob.glob(KOK + '/lightning_logs/**/checkpoints/last.ckpt', recursive=True), key=os.path.getmtime)
if son:
    baslangic = '--ckpt_path "%s"' % son[-1]; print('Sürdürülüyor:', son[-1])
elif BASLAT == 'ckpt':
    baslangic = '--ckpt_path "%s"' % CKPT
else:
    baslangic = '--model.vocoder_warmstart_ckpt "%s"' % CKPT
AZAMI = 5680 + EK_TUR if BASLAT == 'ckpt' or son else EK_TUR
!python3 -m piper.train fit \
  --data.voice_name "$SES_ADI" \
  --data.csv_path /content/veri/metadata.csv \
  --data.audio_dir /content/veri/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice tr \
  --data.cache_dir /content/onbellek \
  --data.config_path "$KOK/$SES_ADI.onnx.json" \
  --data.batch_size 16 \
  --trainer.max_epochs $AZAMI \
  --trainer.default_root_dir "$KOK" \
  {baslangic}

## 7) Modeli dışa aktar ve dinle

En iyi kayıt otomatik seçilir (en yeni `last.ckpt`). Kulağınıza göre başka bir kayıt seçmek isterseniz `SECILEN` yolunu değiştirin.

In [ ]:
SECILEN = sorted(glob.glob(KOK + '/lightning_logs/**/checkpoints/last.ckpt', recursive=True), key=os.path.getmtime)[-1]
print(SECILEN)
!python3 -m piper.train.export_onnx --checkpoint "$SECILEN" --output-file "$KOK/$SES_ADI.onnx"
!echo 'Merhaba, ben Ülgen. Artık sizin sesinizle konuşuyorum.' | python3 -m piper -m "$KOK/$SES_ADI.onnx" -f /content/deneme.wav
from IPython.display import Audio; Audio('/content/deneme.wav')

In [ ]:
# 8) İki dosyayı indirin ve Mac'te ~/ÜlgenVeri/sesler/ klasörüne koyun
from google.colab import files
files.download(KOK + '/' + SES_ADI + '.onnx')
files.download(KOK + '/' + SES_ADI + '.onnx.json')

Bitti. Ülgen'de **Ayarlar → Ses → Ülgen'in kendi sesi** listesinde `tr_TR-hakan-medium (kendi sesiniz)` görünür; seçip **Bu sesle oku: açık** yapın.